# Notebook 03 — Alignment

**Goal:** Pair each audio segment with its correct text label and build the
HuggingFace `DatasetDict` used by training.

**Why this notebook exists:**
Notebook 01 produced WAV segments and Notebook 02 produced text labels.  This
notebook is the glue layer.  It validates that every segment has a matching
label, checks audio length sanity (too short → background noise, too long →
failed segmentation), and writes the final `DatasetDict` to disk.  Separating
this from training means we can re-run alignment without re-running training.

**Output:** `data/processed/hf_dataset/`

## Step 0 — Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import sys
sys.path.insert(0, '/content/drive/MyDrive/iqraa-ai')
!pip install -q -r /content/drive/MyDrive/iqraa-ai/requirements.txt

## Step 1 — Configure paths

In [ ]:
from pathlib import Path

REPO_ROOT     = Path('/content/drive/MyDrive/iqraa-ai')
SEGMENTS_DIR  = REPO_ROOT / 'data' / 'processed' / 'segments'
LABELS_FILE   = REPO_ROOT / 'data' / 'processed' / 'labels.json'
DATASET_OUT   = REPO_ROOT / 'data' / 'processed' / 'hf_dataset'

# Sanity-check thresholds
MIN_DURATION_S = 0.5   # segments shorter than this are probably silence
MAX_DURATION_S = 30.0  # segments longer than this suggest failed splitting

## Step 2 — Validate segment ↔ label alignment

We compare the set of WAV filenames against the set of label keys.  Any
mismatch is printed so we can fix the segmentation before wasting training time
on misaligned data.

In [ ]:
import json

with open(LABELS_FILE, encoding='utf-8') as f:
    labels = json.load(f)

wav_ids  = {p.stem for p in SEGMENTS_DIR.glob('*.wav')}
label_ids = set(labels.keys())

only_wav   = wav_ids - label_ids
only_label = label_ids - wav_ids
matched    = wav_ids & label_ids

print(f'Matched pairs  : {len(matched)}')
print(f'WAV only (orphan segments)  : {len(only_wav)}')
print(f'Label only (missing audio)  : {len(only_label)}')

if only_wav:
    print('  First 5 orphan WAVs:', sorted(only_wav)[:5])
if only_label:
    print('  First 5 missing audio:', sorted(only_label)[:5])

## Step 3 — Duration sanity check

Filter out segments that are too short (likely noise) or too long
(likely a failed split containing multiple ayahs).

In [ ]:
import soundfile as sf
from tqdm import tqdm

valid_ids = []
rejected  = []

for seg_id in tqdm(sorted(matched), desc='Checking durations'):
    wav_path = SEGMENTS_DIR / f'{seg_id}.wav'
    info     = sf.info(str(wav_path))
    duration = info.frames / info.samplerate

    if duration < MIN_DURATION_S or duration > MAX_DURATION_S:
        rejected.append((seg_id, round(duration, 2)))
    else:
        valid_ids.append(seg_id)

print(f'Valid   : {len(valid_ids)}')
print(f'Rejected: {len(rejected)}')
if rejected:
    print('  First 5 rejected:', rejected[:5])

## Step 4 — Build and save the HuggingFace DatasetDict

We call `src.dataset.build_dataset()` which wraps the validated segments into
a `DatasetDict` with `train` and `test` splits, then saves to disk.  The saved
dataset is loaded directly by Notebook 04 (training).

In [ ]:
from src.dataset import build_dataset, save_dataset
import yaml

with open(REPO_ROOT / 'configs' / 'training_config.yaml') as f:
    cfg = yaml.safe_load(f)

ds = build_dataset(
    segments_dir=str(SEGMENTS_DIR),
    labels_path=str(LABELS_FILE),
    test_split=cfg['data']['test_split'],
)

print(ds)
save_dataset(ds, str(DATASET_OUT))
print(f'Dataset saved to {DATASET_OUT}')